In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anirudhtn/commonnames/Names from Makemore.txt


In [3]:
import torch
import matplotlib.pyplot as plt
import random
import math
import torch.nn.functional as F

In [4]:
words = open('/kaggle/input/datasets/anirudhtn/commonnames/Names from Makemore.txt', 'r').read().splitlines()
print(len(words))

32033


In [5]:
chars = sorted(set("".join(words)))
chars
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [6]:
# use the first 3 characters to predict the fourth character

block_size = 3 

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtrain,  Ytrain  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xtest,  Ytest  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [7]:
# list of parameters

n_embed = 10 # 10 dimensional vectors to represent each character
n_hidden = 64 # number of neurons in the hidden layer

g = torch.Generator().manual_seed(2147483647)

C = torch.randn(vocab_size, n_embed, generator=g)

# first layer
# using kaiming init to divide by sqrt(fan in)
# and multiplying by 5/3 to get some gain amidst squashing
W1 = torch.randn((n_embed * block_size, n_hidden), generator=g) * (5/3) / ((n_embed * block_size)**0.5)
# you technically don't need bias since you are going to batch norm
b1 = torch.randn(n_hidden, generator=g) * 0.1

# output layer - layer 2
# can use kaiming init if we want
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# batch normalisation params
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1


parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True


4137


In [8]:
batch_size = 32

ix = torch.randint(0, Xtrain.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtrain[ix], Ytrain[ix]
Xb.shape

torch.Size([32, 3])

In [9]:
embedded = C[Xb]
emb_view = embedded.view(embedded.shape[0], -1)
emb_view.shape


#forward pass

# pre batch-norm
h_pre_bn = emb_view @ W1 + b1
h_pre_bn.shape

print(h_pre_bn.shape)
# batch-norm
bnmeani = 1/batch_size * h_pre_bn.sum(0, keepdim=True)
print(bnmeani.shape)
bndiff = h_pre_bn - bnmeani
bndiff2 = bndiff**2

# use bessel's correction to sample the variance
bnvar = 1/(batch_size - 1) * bndiff2.sum(0, keepdim=True)

# 1 / sqrt(var + eps)
# eps is used to prevent division by 0 when there is no variance
bnvar_inv = (bnvar + 1e-5)**-0.5 

bnraw = bndiff * bnvar_inv

h_pre_act = bngain * bnraw + bnbias

# non-linearity tanh
h = torch.tanh(h_pre_act)

# layer 2 
logits = h @ W2 + b2

# implementing cross entropy loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()



counts_sum = counts.sum(1, keepdims = True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv

logprobs = probs.log()
loss = -logprobs[range(batch_size), Yb].mean()

# backward pass

for p in parameters:
    p.grad = None

for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logit_maxes, logits, h, h_pre_act, bnraw,bnvar_inv, bnvar, bndiff2, bndiff, h_pre_bn, bnmeani,emb_view, embedded]:
    t.retain_grad()

loss.backward()
loss

torch.Size([32, 64])
torch.Size([1, 64])


tensor(3.3314, grad_fn=<NegBackward0>)

In [10]:
# defining a comparator function that checks the loss calculated 
# by us v/s the loss calculated by pyTorch

def cmp(variable, by_us, by_pytorch):
    ex = torch.all(by_us == by_pytorch.grad).item()
    approx = torch.allclose(by_us, by_pytorch.grad)
    max_diff = (by_us - by_pytorch.grad).abs().max().item()

    print(f'{variable:10s} | exact: {str(ex):5s} | approx: {str(approx):5s} | max_diff: {str(max_diff):5s}')

In [11]:
# dlogprobs - derivate of loss wrt all elements of logprobs
# dL/dlogprobs

logprobs.shape

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(batch_size), Yb] = -1.0/batch_size
logprobs.shape

cmp('logprobs', dlogprobs, logprobs)

logprobs   | exact: True  | approx: True  | max_diff: 0.0  


In [12]:
# logprobs = probs.log()
# dloss/dprobs = dloss/dlogprobs * dlogprobs/dprobs
# dlogprobs/dprobs = 1/each element

dprobs = torch.zeros_like(probs)
dprobs.shape == probs.shape

dprobs[range(batch_size), Yb] = 1.0/probs[range(batch_size), Yb]
dprobs = dprobs * dlogprobs

cmp('probs', dprobs, probs)

probs      | exact: True  | approx: True  | max_diff: 0.0  


In [13]:
counts_sum_inv.shape

# we want to find dLoss/dcounts_sum_inv = dcounts_sum_inv
# dcounts_sum_inv is going to be a 1x32 tensor that is going to be broadcasted
# across all columns. So basically, there are 2 operations that happen in the forward pass
# 1. replicate the column tensor across all columns
# 2. multiply counts * count_sum _inv
# dLoss/dcount_sum_inv = dLoss/dprobs * dprobs/dcount_sum_inv

# probs = counts * counts_sum_inv
# so, dprobs/dcount_sum_inv = counts
# but the count is now replicated to all columns, so sum that up

dcounts_sum_inv = (counts * dprobs).sum(1, keepdims=True)

# similarly, dcounts is just counts_sum_inv x dprobs

dcounts = counts_sum_inv * dprobs

dcounts.shape
dcounts_sum_inv.shape

# cmp('dcounts', dcounts, counts)
cmp('dcounts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum_inv | exact: True  | approx: True  | max_diff: 0.0  


In [14]:
# counts_sum_inv = counts_sum**-1
# dLoss/dcounts_sum = dcounts_sum_inv/dcounts_sum * dLoss/dcounts_sum_inv

dcounts_sum = -(counts_sum**-2) * dcounts_sum_inv

cmp('dcounts_sum', dcounts_sum, counts_sum)


dcounts_sum | exact: True  | approx: True  | max_diff: 0.0  


In [15]:
counts.shape, counts_sum.shape
# take care of the derivative of the second counts component 
# counts_sum = counts.sum(1, keepdims = True)
# here, counts_sum is a column tensor that has all sums calculated. 
# Assume counts is a and counts sum is b 
# a11 + a12 + a13 = b1
# now, you already have db1 which is dcounts_sum.
# the grad just flows through all elements equally since its addition
#d/dx will just give you 1



dcounts += torch.ones_like(counts) * dcounts_sum

cmp('dcounts', dcounts, counts)



dcounts    | exact: True  | approx: True  | max_diff: 0.0  


In [16]:
# dnorm_logits
# counts = norm_logits.exp()

#d/dx of e^x = e^x

# dnorm_logits = norm_logits.exp() * dcounts
# but norm_logits.exp() = counts

dnorm_logits = counts * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)


norm_logits | exact: True  | approx: True  | max_diff: 0.0  


In [17]:
# norm_logits = logits - logit_maxes
norm_logits.shape, logits.shape, logit_maxes.shape
dlogits = dnorm_logits.clone()
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)

cmp('dlogit_maxes', dlogit_maxes, logit_maxes)

dlogit_maxes | exact: True  | approx: True  | max_diff: 0.0  


In [18]:
dlogit_maxes

#dlogit_maxes has no effect on loss - so the gradient must be almost 0

# we are now computing the second component of dlogits
# logits has an effect of logit_maxes - we take the max elements from 
# all of the row tensors and only toggle that in the output
# so basically, only that element times the dlogit_maxes will give you
# the differential here

dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes


cmp('dlogits', dlogits, logits)

dlogits    | exact: True  | approx: True  | max_diff: 0.0  


In [19]:
# differentiating the matrix multiplication
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

h          | exact: True  | approx: True  | max_diff: 0.0  
W2         | exact: True  | approx: True  | max_diff: 0.0  
b2         | exact: True  | approx: True  | max_diff: 0.0  


In [20]:
# differentiating the tanh
dh_pre_act = (1.0 - h**2) * dh
cmp('dh_pre_act', dh_pre_act, h_pre_act)

dh_pre_act | exact: False | approx: True  | max_diff: 4.656612873077393e-10


In [21]:
# bnraw = bndiff * bnvar_inv
# h_pre_act = bngain * bnraw + bnbias
# finding out how h_pre_act differs with bngain, bnraw and bnbias

# dh_pre_act/dbngain = bnraw * dh_pre_act

dbngain = (bnraw * dh_pre_act).sum(0, keepdim=True)

cmp('dbngain', dbngain, bngain)

# dbnraw = bngain * dh_pre_act

dbnraw = bngain * dh_pre_act
dbnbias = dh_pre_act.sum(0, keepdim=True)

cmp('dbngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('dbnbias', dbnbias, bnbias)


dbngain    | exact: False | approx: True  | max_diff: 1.862645149230957e-09
dbngain    | exact: False | approx: True  | max_diff: 1.862645149230957e-09
bnraw      | exact: False | approx: True  | max_diff: 4.656612873077393e-10
dbnbias    | exact: False | approx: True  | max_diff: 5.587935447692871e-09


In [27]:
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv

# cmp('dbnbdiff', dbnbdiff, bnbdiff)
cmp('dbnvar_inv', dbnvar_inv, bnvar_inv)
cmp('dbnvar', dbnvar, bnvar)

dbnvar_inv | exact: False | approx: True  | max_diff: 3.725290298461914e-09
dbnvar     | exact: False | approx: True  | max_diff: 6.984919309616089e-10


In [28]:
dbndiff2 = (1.0/(batch_size-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += (2*bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)



In [32]:
dhprebn += 1.0/batch_size * (torch.ones_like(h_pre_bn) * dbnmeani)

In [38]:
dembview = dhprebn @ W1.T
dW1 = emb_view.T @ dhprebn
db1 = dhprebn.sum(0)

In [42]:
demb = dembview.view(embedded.shape)

In [43]:
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]

In [48]:
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, h_pre_bn)
cmp('embcat', dembview, emb_view)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, embedded)
cmp('C', dC, C)

bndiff2    | exact: False | approx: True  | max_diff: 2.1827872842550278e-11
bndiff     | exact: False | approx: True  | max_diff: 4.656612873077393e-10
bnmeani    | exact: False | approx: True  | max_diff: 2.7939677238464355e-09
hprebn     | exact: False | approx: True  | max_diff: 4.656612873077393e-10
embcat     | exact: False | approx: True  | max_diff: 1.862645149230957e-09
W1         | exact: False | approx: True  | max_diff: 4.6566128730773926e-09
b1         | exact: False | approx: True  | max_diff: 2.7939677238464355e-09
emb        | exact: False | approx: True  | max_diff: 1.862645149230957e-09
C          | exact: False | approx: True  | max_diff: 7.450580596923828e-09


In [49]:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3313660621643066 diff: 2.384185791015625e-07


In [51]:
# simplifying the backward  pass

dlogits = F.softmax(logits, 1)
dlogits[range(batch_size), Yb] -= 1
dlogits /= batch_size

cmp('logits', dlogits, logits)


logits     | exact: False | approx: True  | max_diff: 7.683411240577698e-09
